# Làm sạch giao dịch và tính RFM

Áp dụng quy tắc trong `docs/data_rules.md` cho `kz.csv`. Notebook tạo cùng đầu ra với `scripts/run_pipeline.py`.

## 1. Áp dụng quy tắc và lưu Parquet

Với dòng 6 trường khớp mẫu, đưa giá và mã khách về đúng cột. PySpark kiểm tra mã, ngày và giá.
Lưu dòng hợp lệ và tách dòng bị loại kèm lý do.

In [1]:
from pathlib import Path
import sys

from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

SOURCE = ROOT / "data/raw/kz.csv"
REPORTS = ROOT / "reports"

from src.prepare_data import prepare_data, start_spark
from src.rfm import check_rfm, save_rfm

OUTPUT = ROOT / "data/processed"
spark = start_spark()
spark.conf.set("spark.sql.repl.eagerEval.enabled", "true")
spark.conf.set("spark.sql.repl.eagerEval.truncate", "0")

clean, cleaning = prepare_data(spark, SOURCE, OUTPUT, REPORTS)

metrics = ["raw_rows", "clean_rows", "quarantine_rows", "accepted_repaired_rows"]
counts = [(metric, cleaning[metric]) for metric in metrics]
display(spark.createDataFrame(counts, ["metric", "rows"]))

reasons = list(cleaning["rows_by_primary_reason"].items())
display(spark.createDataFrame(reasons, ["primary_reason", "rows"]))

metric,rows
raw_rows,2633521
clean_rows,600784
quarantine_rows,2032737
accepted_repaired_rows,37961


primary_reason,rows
nonpositive_price,568
missing_customer_id,2030733
date_outside_scope,1436
accepted,600784


## 2. Kiểm tra đầu ra làm sạch

Xem mẫu dòng 6 trường đã sửa và bảng tỷ lệ thiếu trước–sau.
`before_validation` là dữ liệu đã căn cột nhưng chưa tách dòng không hợp lệ.

In [2]:
repaired_sample = clean.filter("row_status = 'repaired_6_fields'").select(
    "source_record", "event_time", "order_id", "category_id", "brand", "price", "customer_id"
).orderBy("source_record").limit(3)
display(repaired_sample)

quality = spark.read.option("header", True).csv(str(REPORTS / "cleaning_quality.csv"))
display(quality)

source_record,event_time,order_id,category_id,brand,price,customer_id
1813,2020-05-02 11:12:20,2300138854784434309,NULL,NULL,0.0200,1515915625453675584
3634,2020-05-06 07:37:55,2302930038149874117,NULL,NULL,38.1700,1515915625455624044
3919,2020-05-06 22:36:13,2303382163636093788,NULL,NULL,0.0200,1515915625441101795


stage,column,rows,missing_count,missing_pct
before_validation,event_time,2633521,0,0.0
before_validation,order_id,2633521,0,0.0
before_validation,product_id,2633521,0,0.0
before_validation,category_id,2633521,431954,16.4021
before_validation,category_code,2633521,1044156,39.6487
before_validation,brand,2633521,544624,20.6805
before_validation,price,2633521,0,0.0
before_validation,user_id,2633521,2030733,77.1109
after_validation,event_time,600784,0,0.0
after_validation,order_id,600784,0,0.0


## 3. Tạo bảng RFM và đối chiếu

Mỗi khách có một dòng. R là số ngày từ lần mua cuối đến ngày chốt, F là số đơn riêng biệt,
M là tổng giá các dòng mua hợp lệ. Tính lại ba mẫu từ các dòng mua sạch để kiểm tra công thức.

In [3]:
try:
    rfm = save_rfm(clean, OUTPUT, REPORTS)
    metrics = [
        "reference_date", "customers", "orders", "repeat_customers",
        "repeat_customer_pct", "monetary_value",
    ]
    summary = [(metric, str(rfm[metric])) for metric in metrics]
    display(spark.createDataFrame(summary, ["metric", "value"]))

    customer_rfm = spark.read.parquet(str(OUTPUT / "customer_rfm.parquet"))
    display(customer_rfm.limit(5))

    check_rfm(spark, clean, OUTPUT, REPORTS)
    rfm_checks = spark.read.option("header", True).csv(str(REPORTS / "rfm_hand_check.csv"))
    display(rfm_checks)
finally:
    clean.unpersist()
    spark.stop()

metric,value
reference_date,2020-11-22
customers,234359
orders,402993
repeat_customers,74783
repeat_customer_pct,31.9096
monetary_value,118372961.6200


customer_id,recency_days,frequency_orders,monetary_value,last_purchase_at
1515915625447879434,32,3,2841.2900,2020-10-21 19:40:23
1515915625451131565,209,1,274.2300,2020-04-27 14:57:22
1515915625442675260,134,4,2337.6100,2020-07-11 04:19:59
1515915625452188656,207,1,20.8100,2020-04-29 12:10:33
1515915625451952331,207,1,20.8100,2020-04-29 13:14:42


customer_id,manual_R,saved_R,manual_F,saved_F,manual_M,saved_M,matched
1515915625439952115,136,136,1,1,416.6400,416.6400,True
1515915625453675584,204,204,1,1,9.2600,9.2600,True
1515915625514704541,4,4,178,178,39878.7200,39878.7200,True
